# Stock histórico y consumo mensual por material/centro

**Fuente:** Athena `molinos_datalake_datapreparation.sap_pp_docordpro`  
**Lógica de stock:** stock inicial (CSV externo al 2024-12-01) + movimientos reales acumulados mes a mes  
**Lógica de consumo:** movimiento 261 (salida para orden de producción)  

Movimientos excluidos del cálculo de stock: traslados y reclasificaciones internas (311, 312, 301, 302, 321, 323, 343, 344, 641, 671, etc.) porque solo aparece una punta en la tabla.

## 1. Setup

In [ ]:
import boto3, time
from dotenv import dotenv_values
from datetime import date
import pandas as pd

config = dotenv_values(".env")

athena = boto3.client(
    "athena",
    region_name=config["AWS_REGION_NAME"],
    aws_access_key_id=config["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=config["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=config["AWS_SESSION_TOKEN"],
)
ATHENA_DB     = "molinos_datalake_datapreparation"
ATHENA_OUTPUT = config["ATHENA_OUTPUT_LOCATION"]

def run_athena(sql: str) -> pd.DataFrame:
    r = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": ATHENA_DB},
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT},
    )
    qid = r["QueryExecutionId"]
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if status in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)
    if status != "SUCCEEDED":
        raise Exception(athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"])
    rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    headers = [d["VarCharValue"] for d in rows[0]["Data"]]
    data    = [[d.get("VarCharValue", "") for d in r["Data"]] for r in rows[1:]]
    return pd.DataFrame(data, columns=headers)

print("Setup OK")

## 2. Parámetros

In [ ]:
# Fecha del stock inicial que provee el usuario
FECHA_INICIO = date(2024, 12, 1)

# Movimientos que representan flujos reales de entrada/salida de stock.
# Se excluyen traslados y reclasificaciones (311, 301, 321, 641, etc.)
# porque solo tienen una punta registrada en la tabla.
MOVIMIENTOS_REALES = (
    "101", "102",   # Entrada de mercancías (compras) y su anulación
    "105", "106",   # EM/DM desde stock bloqueado
    "122", "123",   # Devolución a proveedor y anulación
    "161", "162",   # EM devolución / anulación devolución
    "261",          # Consumo para orden de producción (a agregar)
    "501", "502",   # Entrada/salida sin pedido
    "521", "522",   # Entrada/salida sin orden de fabricación
    "531", "532",   # Entrada/salida subproducto
    "545", "546",   # Entrada/salida subproducto subcontratación
    "561", "562",   # Entrada/salida inicial de stocks
    "631", "632",   # Stock en consignación
    "651", "652",   # Entrega mercancía devolución
    "653", "654",   # EM/DM devolución libre utilización
    "657", "658",   # EM/DM devolución bloqueada
)

filtro_mov = ", ".join(f"'{m}'" for m in MOVIMIENTOS_REALES)

print(f"Fecha inicio: {FECHA_INICIO}")
print(f"Movimientos incluidos: {len(MOVIMIENTOS_REALES)}")

## 3. Stock al 1° de cada mes

### 3.1 Stock inicial (CSV externo al 2024-12-01)

Formato del CSV exportado desde AFO/BW: separador `;`, números con formato europeo (`.` miles, `,` decimal), cantidad y unidad en una sola columna.

In [ ]:
df_raw = pd.read_csv(
    "stock_inicial_20241201.csv",
    encoding="latin1",
    sep=";",
    dtype=str,
)

# Renombrar columnas
df_raw.columns = ["material", "descripcion_material", "centro", "descripcion_centro", "cantidad_unidad"]

# Separar cantidad y unidad (ej: "12.405,751 KG" → 12405.751, "KG")
df_raw[["cantidad_str", "unidad"]] = df_raw["cantidad_unidad"].str.rsplit(" ", n=1, expand=True)

# Formato europeo → float: quitar punto de miles, reemplazar coma decimal por punto
df_raw["cantidad_stock"] = (
    df_raw["cantidad_str"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
)

df_inicial = df_raw[["material", "centro", "cantidad_stock", "unidad"]].copy()
df_inicial["material"] = df_inicial["material"].astype(str).str.strip()
df_inicial["centro"]   = df_inicial["centro"].astype(str).str.strip()

print(f"Stock inicial: {len(df_inicial):,} filas | {df_inicial['material'].nunique()} materiales | {df_inicial['centro'].nunique()} centros")
df_inicial.head()

### 3.2 Movimientos netos por material/centro/mes desde Athena

In [ ]:
df_mov = run_athena(f"""
    SELECT
        sku                                              AS material,
        planta                                           AS centro,
        DATE_TRUNC('month', fecha_contabilizacion)       AS mes,
        unidad_base_material                             AS unidad,
        SUM(cantidad_produccion_umb)                     AS movimiento_neto
    FROM sap_pp_docordpro
    WHERE codigo_tipo_movimiento_mercancia IN ({filtro_mov})
      AND fecha_contabilizacion >= DATE '{FECHA_INICIO}'
    GROUP BY
        sku,
        planta,
        DATE_TRUNC('month', fecha_contabilizacion),
        unidad_base_material
    ORDER BY material, centro, mes
""")

df_mov["mes"]             = pd.to_datetime(df_mov["mes"]).dt.date
df_mov["movimiento_neto"] = pd.to_numeric(df_mov["movimiento_neto"], errors="coerce").fillna(0)
df_mov["material"]        = df_mov["material"].astype(str)
df_mov["centro"]          = df_mov["centro"].astype(str)

print(f"Movimientos: {len(df_mov):,} filas | {df_mov['mes'].nunique()} meses | {df_mov['material'].nunique()} materiales")
df_mov.head()

### 3.3 Reconstrucción del stock mes a mes

In [ ]:
# Todos los meses disponibles en los movimientos
meses = sorted(df_mov["mes"].unique())

# Todas las combinaciones material/centro (unión de inicial + movimientos)
combos = pd.concat([
    df_inicial[["material", "centro", "unidad"]],
    df_mov[["material", "centro", "unidad"]],
]).drop_duplicates(subset=["material", "centro"])

# Grid completo: cada combo × cada mes
grid = combos.assign(_key=1).merge(
    pd.DataFrame({"mes": meses, "_key": 1}),
    on="_key"
).drop(columns="_key")

# Pegar movimientos
grid = grid.merge(
    df_mov[["material", "centro", "mes", "movimiento_neto"]],
    on=["material", "centro", "mes"],
    how="left",
).fillna({"movimiento_neto": 0})

# Pegar stock inicial
grid = grid.merge(
    df_inicial[["material", "centro", "cantidad_stock"]].rename(columns={"cantidad_stock": "stock_inicial"}),
    on=["material", "centro"],
    how="left",
).fillna({"stock_inicial": 0})

grid = grid.sort_values(["material", "centro", "mes"]).reset_index(drop=True)

# Stock al 1° del mes = stock_inicial + cumsum de movimientos de meses ANTERIORES (shift 1 dentro de cada grupo)
grid["stock_inicio_mes"] = (
    grid["stock_inicial"]
    + grid.groupby(["material", "centro"])["movimiento_neto"]
          .transform(lambda x: x.cumsum().shift(1, fill_value=0))
)

df_stock = grid[["material", "centro", "mes", "stock_inicio_mes", "unidad"]].copy()

# Agregar la foto del stock inicial (mes FECHA_INICIO)
df_stock_ini = df_inicial.rename(columns={"cantidad_stock": "stock_inicio_mes"}).assign(mes=FECHA_INICIO)
df_stock = pd.concat([df_stock_ini[["material","centro","mes","stock_inicio_mes","unidad"]], df_stock], ignore_index=True)
df_stock = df_stock.sort_values(["material", "centro", "mes"]).reset_index(drop=True)

print(f"Stock reconstruido: {len(df_stock):,} filas | {df_stock['mes'].nunique()} meses")
df_stock.head(10)

## 4. Consumo mensual (mov. 261)

> Pendiente: que el equipo de datos agregue el movimiento `261` a la tabla. La consulta ya está lista.

In [ ]:
df_consumo = run_athena(f"""
    SELECT
        sku                                              AS material,
        planta                                           AS centro,
        DATE_TRUNC('month', fecha_contabilizacion)       AS mes,
        unidad_base_material                             AS unidad,
        ABS(SUM(cantidad_produccion_umb))                AS cantidad_consumo
    FROM sap_pp_docordpro
    WHERE codigo_tipo_movimiento_mercancia = '261'
      AND fecha_contabilizacion >= DATE '{FECHA_INICIO}'
    GROUP BY
        sku,
        planta,
        DATE_TRUNC('month', fecha_contabilizacion),
        unidad_base_material
    ORDER BY material, centro, mes
""")

df_consumo["mes"]              = pd.to_datetime(df_consumo["mes"]).dt.date
df_consumo["cantidad_consumo"] = pd.to_numeric(df_consumo["cantidad_consumo"], errors="coerce").fillna(0)
df_consumo["material"]         = df_consumo["material"].astype(str)
df_consumo["centro"]           = df_consumo["centro"].astype(str)

print(f"Consumo: {len(df_consumo):,} filas | {df_consumo['mes'].nunique()} meses | {df_consumo['material'].nunique()} materiales")
df_consumo.head()

## 5. Consolidación y salida

In [ ]:
df_final = df_stock.merge(
    df_consumo[["material", "centro", "mes", "cantidad_consumo"]],
    on=["material", "centro", "mes"],
    how="left",
)

df_final = df_final.sort_values(["material", "centro", "mes"]).reset_index(drop=True)

out_path = "stock_consumo_mensual.csv"
df_final.to_csv(out_path, index=False)
print(f"Listo. {len(df_final):,} filas → {out_path}")
df_final.head(10)